In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import os, sys, re, random, time
import importlib
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import faiss

import data_loaders
import miscellaneous
from rs_algorithm.models import RegressionBasedBPEModel
from rs_algorithm.models import AdaptiveRepresentation
from dl_trainer import Trainer, split_collate

In [5]:
os.getcwd()

'C:\\Beryllium Base\\WORKSPACES\\MTECH\\RS-project\\sandbox'

In [6]:
data_model = data_loaders.KuaiRecData(root='../data/KuaiRec2.0', initial_specimen='smaller')

In [7]:
data_model.ratings.head()

,user_id,item_id,rating,time
0,1520,1357,0.471552,2020-08-19 23:02:14.022
1,4984,8432,0.427609,2020-07-13 16:27:10.746
2,6474,3934,0.851681,2020-07-24 05:24:54.063
3,6435,6264,0.166174,2020-08-18 09:19:47.704
4,1094,8648,0.784189,2020-07-28 18:35:20.47


In [12]:
try:
    ds_train = data_model.get_bpr_dataset()
    smaller_matrix = miscellaneous.keyed_cache("kuairec_smaller_matrix", lambda: data_model.subsample_specimen('small', 10000))
    ds_test = data_model.get_bpr_dataset(smaller_matrix)
except:
    traceback.print_exc()

In [13]:
try:
    d_embed = 1024
    n_users, d_user_feats = data_model.user_features.shape
    n_items, d_item_feats = data_model.item_features.shape
    # n_users 7176 d_user_feats 1592
    # n_items 10728 d_item_feats 429

    n_users, d_user_feats = data_model.ratings['user_id'].nunique(), data_model.user_features.shape[1]
    n_items, d_item_feats = data_model.ratings['item_id'].nunique(), data_model.item_features.shape[1]
    # n_users 1411 d_user_feats 1592
    # n_items 3327 d_item_feats 429

    uid2idx = {v: i for i, v in enumerate(np.unique(data_model.ratings['user_id']))}
    user_model = AdaptiveRepresentation(n_users, d_user_feats, d_hidden=1024, d_embed=d_embed, id2idx=uid2idx)
    iid2idx = {v: i for i, v in enumerate(np.unique(data_model.ratings['item_id']))}
    item_model = AdaptiveRepresentation(n_items, d_item_feats, d_hidden=1024, d_embed=d_embed, id2idx=iid2idx)
    model = RegressionBasedBPEModel(user_model, item_model)
except:
    traceback.print_exc()

In [14]:
try:
    trainer = Trainer(model=model, train_dataloader=DataLoader(ds_train, batch_size=8192))
except:
    traceback.print_exc()

In [15]:
try:
    trainer.load_checkpoint()
except:
    traceback.print_exc()

Pick an option
  0> View All
  1> 1st Last Model checkpoint_RegressionBasedBPEModel_load_backup_20260405_055236_None.pt
  2> 2st Last Model checkpoint_RegressionBasedBPEModel_load_backup_20260405_053749_None.pt
  3> 3st Last Model checkpoint_RegressionBasedBPEModel_001_20260402_171557_0.7496953347667319.pt


In [143]:
try:
    with torch.no_grad():
        index = faiss.IndexFlatIP(d_embed)
        for iid in smaller_matrix['item_id'].unique():
            ift = data_model.item_features.iloc[iid].to_numpy(dtype=np.float32)
            iid = torch.tensor([iid], dtype=torch.int64)
            ift = torch.tensor(ift[None, :], dtype=torch.float32)
            ie, *_ = item_model(iid, ift)
            index.add(ie)
except:
    traceback.print_exc()

In [144]:
try:
    print(index)
except:
    traceback.print_exc()

<faiss.swigfaiss_avx2.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x000002696A73C870> >


In [151]:
try:
    with torch.no_grad():
        (uid, uft, pid, pif, nid, nif), truth = next(iter(ds_test))
        uid, uft, pid, pif, nid, nif = (
            torch.tensor([uid], dtype=torch.int64), torch.tensor(uft[None, :], dtype=torch.float32),
            torch.tensor([pid], dtype=torch.int64), torch.tensor(pif[None, :], dtype=torch.float32),
            torch.tensor([nid], dtype=torch.int64), torch.tensor(nif[None, :], dtype=torch.float32))
        ue, ub = user_model(uid, uft)
        pie, pib = item_model(pid, pif)
        nie, nib = item_model(nid, nif)
        print((ue * pie).sum())
        print((ue * nie).sum())

        scores, indices = index.search(ue, 10)
        for i in indices[0]:
            (uid, uft, pid, pif, nid, nif), truth = ds_test[i]
            uid, uft, pid, pif, nid, nif = (
                torch.tensor([uid], dtype=torch.int64), torch.tensor(uft[None, :], dtype=torch.float32),
                torch.tensor([pid], dtype=torch.int64), torch.tensor(pif[None, :], dtype=torch.float32),
                torch.tensor([nid], dtype=torch.int64), torch.tensor(nif[None, :], dtype=torch.float32))
            ue2, ub = user_model(uid, uft)
            print((ue * ue2).sum())
            # break
except:
    traceback.print_exc()

tensor(57.2656)
tensor(-49.9655)
tensor(264.7013)
tensor(272.6363)
tensor(257.2510)
tensor(230.5111)
tensor(263.7266)
tensor(270.2513)
tensor(252.7347)
tensor(243.3621)
tensor(260.7346)
tensor(251.1512)
